In [18]:
import pandas as pd
import regex as re

In [19]:
# For the presentation: created a diff function to count the "number of operations" that we made on the data.
# Created with Copilot
import numpy as np

def safe_equal(a, b):
    """Return True if values are equal, handling lists, Series, NaN, etc."""
    
    # Handle NaN
    if isinstance(a, float) and isinstance(b, float):
        if np.isnan(a) and np.isnan(b):
            return True

    # Convert Series to list for comparison
    if isinstance(a, pd.Series):
        a = a.tolist()
    if isinstance(b, pd.Series):
        b = b.tolist()

    # Convert numpy arrays to lists
    if hasattr(a, "tolist") and not isinstance(a, str):
        a = a.tolist()
    if hasattr(b, "tolist") and not isinstance(b, str):
        b = b.tolist()

    # Now compare using Python
    return a == b

def df_diff(before, after):
    """Full diff between two dataframes, safe for explode, Series cells, lists, etc."""

    shared_cols = before.columns.intersection(after.columns)

    removed_idx = before.index.difference(after.index)
    added_idx = after.index.difference(before.index)

    removed_rows = before.loc[removed_idx] if len(removed_idx) else pd.DataFrame()
    added_rows = after.loc[added_idx] if len(added_idx) else pd.DataFrame()

    common_idx = before.index.intersection(after.index)

    before_common = before.loc[common_idx, shared_cols]
    after_common = after.loc[common_idx, shared_cols]

    changes = []

    for idx in common_idx:
        row_before = before_common.loc[idx]
        row_after = after_common.loc[idx]

        for col in shared_cols:
            v1 = row_before[col]
            v2 = row_after[col]

            if not safe_equal(v1, v2):
                changes.append({
                    "index": idx,
                    "column": col,
                    "before": v1,
                    "after": v2,
                    "_change": "cell_changed"
                })

    changes = pd.DataFrame(changes)

    return {
        "added_rows": added_rows.assign(_change="row_added"),
        "removed_rows": removed_rows.assign(_change="row_removed"),
        "cell_changes": changes
    }

def track(data, label):
    global ops
    global last_snapshot

    diff = df_diff(last_snapshot, data)

    print(f"\n=== DIFF AFTER: {label} ===")
    print("Added rows:", len(diff["added_rows"]))
    print("Removed rows:", len(diff["removed_rows"]))
    print("Cell changes:", len(diff["cell_changes"]))

    # You can also save diffs to disk if needed:
    # diff["cell_changes"].to_csv(f"diffs/{label}_cells.csv", index=False)

    ops += len(diff["added_rows"]) + len(diff["removed_rows"]) + len(diff["cell_changes"])
    last_snapshot = data.copy()

In [20]:
data = pd.read_csv("data/translation_data_full_duplicates.csv")

In [21]:
data

,teaviku_laad,sihtrühm,pealkiri,autor,originaal_pealkiri,Žanr,tõlkija,lähtekeel,lähtekirjandus,toimetaja,...,ilmumisaasta,füüsiline_kirjeldus,trükikordus,sarjaandmed,arvustatava_tõlkeraamatu_autor,väljaanne,number_leheküljed,sisu,märkused,id
0,Tõlkearvustus,Täiskasvanute ilukirjandus,Natanael. Kulturhistorialine roman. A. Liepe j...,π,NaN,romaanid,"Meomuttel, Jüri 1868-1948",saksa,saksa,NaN,...,1896,NaN,NaN,NaN,"Liepe, Albert",<p>Olevik</p>,"13. veebr., nr 7, lk 161-162",NaN,Idna Illiv (pseud.) = Jüri Meomuttel,71113
1,Tõlkearvustus,Täiskasvanute ilukirjandus,Talu sulane Tõnis. Ühe Schweitsi jutu järele j...,π,NaN,jutud,teadmata,saksa,šveitsi,NaN,...,1897,NaN,NaN,NaN,"Stael von Holstein, Lucia 1857-1929",<p>Olevik</p>,"28. okt., nr 43, lk 961",NaN,NaN,53196
2,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Miss Portsia,"Zunk, Paul",teadmata,jutud,"Mällow, J.",teadmata,teadmata,NaN,...,1894,NaN,NaN,NaN,NaN,<p>Eesti Postimees : Lisaleht</p>,nr 21-22,NaN,Väljaandes: Paul Zunk'i uudisjutukese järele J...,63698
3,Kogumikus ilmunud tõlge,Täiskasvanute ilukirjandus,Kodu,"Žukovski, Vassili 1783-1852",NaN,luuletused,"Leppik, Jaan 1861-1943",vene,vene,NaN,...,1889,39 lk. 13 cm,NaN,NaN,NaN,NaN,NaN,<p>Ilmunud kogumikus: Eestistatud laulud 2.</p>,Väljaandes: Wene keelest [autor märkimata]; Ee...,58892
4,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Kolm õde,"Žukovski, Vassili 1783-1852",teadmata,jutud,G. Kewade,saksa,vene,NaN,...,1886,NaN,NaN,NaN,NaN,<p>Meelelahutaja</p>,"31. dets., nr. 52, lk. 409",NaN,Väljaandes: Shukowsky järele: G. Kewade,55755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52036,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Peremeheta kohver,"Adamov, Arkadi 1920-1991",Чемодан без хозяина,jutustused,"Jürna, Juhan-Kaspar 1931-1982",vene,vene,NaN,...,19721973,NaN,NaN,NaN,NaN,<p>Noorte Hääl</p>,"30. nov.; 1.-3., 5., 7.-8., 10., 12., 13.-17.,...",NaN,NaN,82166
52037,Perioodikas ilmunud tõlge,Laste- ja noortekirjandus,Nõiatulp : ulmejutt,"Abramov, Sergei 1944-2024",teadmata,jutud,"Maksing, Meta",vene,vene,NaN,...,19801981,NaN,NaN,NaN,NaN,<p>Säde</p>,"12., 16., 26. juuli; 2., 6., 9., 16., 23., 27....",NaN,NaN,78190
52038,Tõlkearvustus,Täiskasvanute ilukirjandus,Zsigmond Móricz: Mudakuld. Romaan. Ungari keel...,A. K.,Sárarany,romaanid,"Murakin, Ants 1892-1975",ungari,ungari,NaN,...,12929,NaN,NaN,Looduse kroonine romaan nr 1,"Móricz, Zsigmond 1879-1942",<p>Päevaleht</p>,"3. veebr., nr 33, lk 8",NaN,NaN,52312
52039,Tõlkeraamat,Täiskasvanute ilukirjandus,Irmgard : hästi põnew romaan,NaN,teadmata,romaanid,"Neumann, Mihkel 1866-1941",teadmata/undetermined,teadmata/undetermined,NaN,...,19231924,23 annet (733 lk.),NaN,NaN,NaN,NaN,NaN,NaN,Kaanel pealkirja täiendandmed: Ülipõnew romaan...,45109


In [22]:
ops = 0
last_snapshot = data.copy(deep=True)
track(data, "load")


=== DIFF AFTER: load ===
Added rows: 0
Removed rows: 0
Cell changes: 0


# Goal: make one big table such that every row of translations is represented, and then all the others as well

## Explode all relevant columns at semicolons

In [23]:
data.columns

Index(['teaviku_laad', 'sihtrühm', 'pealkiri', 'autor', 'originaal_pealkiri',
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'ilmumisaasta',
       'füüsiline_kirjeldus', 'trükikordus', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor', 'väljaanne', 'number_leheküljed',
       'sisu', 'märkused', 'id'],
      dtype='str')

In [24]:
data.trükikordus.unique()

<StringArray>
[                                                                                         nan,
                                                                                '2. tr. 1865',
                                                                                     '2. tr.',
                                                                    '2. tr. [esmatrükk 1885]',
                                                                                 '2. tr 1885',
                                                                                     '3. tr.',
                                                                                 '2 tr. 1867',
                                                                    '2. tr. [esmatrükk 1890]',
                                                                                   '2. trükk',
                                                            '3. trükk, esimesed 1841 ja 1842',
                                    

In [25]:
semicolon_splittable = ['author',
       'genre', 'translator', 'source_language',
       'source_literature', 'editor', 'fore_afterword_author',
       'publication_location', 'publisher', 'series',
       'author_of_translated_book_under_review_to_be_removed']
semicolon_splittable = ['autor', 
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor']
for label in semicolon_splittable:
    data[label] = data[label].str.split(';')
    track(data, f"{label} split")
    data = data.explode(label)
    track(data, f"{label} explode")
    data[label] = data[label].str.strip()
    track(data, f"{label} strip")


=== DIFF AFTER: autor split ===
Added rows: 0
Removed rows: 0
Cell changes: 51443

=== DIFF AFTER: autor explode ===
Added rows: 0
Removed rows: 0
Cell changes: 71729

=== DIFF AFTER: autor strip ===
Added rows: 0
Removed rows: 0
Cell changes: 1008

=== DIFF AFTER: Žanr split ===
Added rows: 0
Removed rows: 0
Cell changes: 52034

=== DIFF AFTER: Žanr explode ===
Added rows: 0
Removed rows: 0
Cell changes: 62104

=== DIFF AFTER: Žanr strip ===
Added rows: 0
Removed rows: 0
Cell changes: 478

=== DIFF AFTER: tõlkija split ===
Added rows: 0
Removed rows: 0
Cell changes: 52001

=== DIFF AFTER: tõlkija explode ===
Added rows: 0
Removed rows: 0
Cell changes: 89606

=== DIFF AFTER: tõlkija strip ===
Added rows: 0
Removed rows: 0
Cell changes: 1783

=== DIFF AFTER: lähtekeel split ===
Added rows: 0
Removed rows: 0
Cell changes: 52033

=== DIFF AFTER: lähtekeel explode ===
Added rows: 0
Removed rows: 0
Cell changes: 54414

=== DIFF AFTER: lähtekeel strip ===
Added rows: 0
Removed rows: 0
Cell 

In [26]:
data = data.reset_index(drop=True)
track(data, "reset_index")


=== DIFF AFTER: reset_index ===
Added rows: 72073
Removed rows: 0
Cell changes: 736145


In [27]:
data.columns

Index(['teaviku_laad', 'sihtrühm', 'pealkiri', 'autor', 'originaal_pealkiri',
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'ilmumisaasta',
       'füüsiline_kirjeldus', 'trükikordus', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor', 'väljaanne', 'number_leheküljed',
       'sisu', 'märkused', 'id'],
      dtype='str')

In [31]:
data.to_csv("data/exploded_data.csv", index=False)

In [32]:
print("\nTOTAL OPERATIONS:", ops)


TOTAL OPERATIONS: 1543160


In [33]:
"""TOTAL OPERATIONS: 1543160"""

'TOTAL OPERATIONS: 1543160'

In [28]:
correct_colindex = ["author", "translator", "editor", "fore_afterword_author", 
                    "title", "title_original", "genre", "source_language", "source_literature",
                   "publisher", "publication_location", "publication_year", "physical_description",
                   "series", "publication_type", "target_audience", "edition", "n_pages", "content",
                   "issue", "notes"]

In [29]:
for col in data.columns:
    if col not in correct_colindex:
        print(f"{col} is not in data.columns")

teaviku_laad is not in data.columns
sihtrühm is not in data.columns
pealkiri is not in data.columns
autor is not in data.columns
originaal_pealkiri is not in data.columns
Žanr is not in data.columns
tõlkija is not in data.columns
lähtekeel is not in data.columns
lähtekirjandus is not in data.columns
toimetaja is not in data.columns
ees__järelsõna_autor is not in data.columns
ilmumiskoht is not in data.columns
kirjastaja is not in data.columns
ilmumisaasta is not in data.columns
füüsiline_kirjeldus is not in data.columns
trükikordus is not in data.columns
sarjaandmed is not in data.columns
arvustatava_tõlkeraamatu_autor is not in data.columns
väljaanne is not in data.columns
number_leheküljed is not in data.columns
sisu is not in data.columns
märkused is not in data.columns
id is not in data.columns


In [30]:
data = data.drop(["id", "author_of_translated_book_under_review_to_be_removed"], axis=1)
track(data, "drop_columns")

KeyError: "['author_of_translated_book_under_review_to_be_removed'] not found in axis"

In [ ]:
correct_colindex

In [ ]:
data = data.loc[:, correct_colindex]
track(data, "reorder")